In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
df = pd.read_csv("mushroom_cleaned.csv")
target = "class"
X = df.drop(columns=target)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train shape: {X_train.shape}; test shape: {X_test.shape}")


Train shape: (4000, 9); test shape: (1000, 9)


## Majority-class baseline

Every candidate model should beat this benchmark.


In [2]:
majority_class = y_train.mode().iloc[0]
majority_predictions = np.full(len(y_test), majority_class)
baseline_results = pd.DataFrame([{
    "model": "Majority class",
    "accuracy": accuracy_score(y_test, majority_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, majority_predictions),
    "f1_poisonous": f1_score(y_test, majority_predictions, pos_label="p", zero_division=0),
}])
baseline_results


,model,accuracy,balanced_accuracy,f1_poisonous
0,Majority class,0.621,0.5,0.0


## PyCaret model comparison

PyCaret compares classification models with cross-validation on the training split. The holdout test set remains untouched until the selected model is evaluated.


In [3]:
from pycaret.classification import add_metric, compare_models, finalize_model, predict_model, pull, setup

from sklearn.metrics import balanced_accuracy_score

pycaret_train = X_train.copy()
pycaret_train[target] = y_train
pycaret_session = setup(
    data=pycaret_train,
    target=target,
    session_id=RANDOM_STATE,
    fold=5,
    numeric_imputation="median",
    categorical_imputation="most_frequent",
    verbose=False,
    html=False,
)
add_metric(
    id="balanced_accuracy",
    name="Balanced Accuracy",
    score_func=balanced_accuracy_score,
    target="pred",
    greater_is_better=True,
)
best_model = compare_models(sort="Balanced Accuracy", n_select=1)
pycaret_comparison_table = pull()
pycaret_comparison_table.head(10)


                                   Model  Accuracy     AUC  Recall   Prec.  \
rf              Random Forest Classifier    0.7750  0.8042  0.7750  0.7810   
xgboost        Extreme Gradient Boosting    0.7607  0.7864  0.7607  0.7576   
et                Extra Trees Classifier    0.7314  0.7657  0.7314  0.7264   
knn               K Neighbors Classifier    0.7279  0.7460  0.7279  0.7250   
dt              Decision Tree Classifier    0.6968  0.6734  0.6968  0.6951   
gbc         Gradient Boosting Classifier    0.7261  0.7544  0.7261  0.7377   
ada                 Ada Boost Classifier    0.6879  0.7016  0.6879  0.6817   
lda         Linear Discriminant Analysis    0.6825  0.6710  0.6825  0.6790   
lr                   Logistic Regression    0.6807  0.6713  0.6807  0.6760   
ridge                   Ridge Classifier    0.6811  0.6710  0.6811  0.6778   
qda      Quadratic Discriminant Analysis    0.6646  0.6873  0.6646  0.6963   
nb                           Naive Bayes    0.6621  0.6409  0.66

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Balanced Accuracy,TT (Sec)
rf,Random Forest Classifier,0.7750,0.8042,0.7750,0.7810,0.7632,0.4897,0.5115,0.7292,0.252
xgboost,Extreme Gradient Boosting,0.7607,0.7864,0.7607,0.7576,0.7539,0.4699,0.4773,0.7254,0.342
et,Extra Trees Classifier,0.7314,0.7657,0.7314,0.7264,0.7239,0.4052,0.4118,0.6943,0.240
knn,K Neighbors Classifier,0.7279,0.7460,0.7279,0.7250,0.7151,0.3859,0.4004,0.6817,1.038
dt,Decision Tree Classifier,0.6968,0.6734,0.6968,0.6951,0.6952,0.3501,0.3510,0.6734,0.082
gbc,Gradient Boosting Classifier,0.7261,0.7544,0.7261,0.7377,0.7003,0.3580,0.3979,0.6622,0.232
ada,Ada Boost Classifier,0.6879,0.7016,0.6879,0.6817,0.6631,0.2758,0.2987,0.6263,0.140
lda,Linear Discriminant Analysis,0.6825,0.6710,0.6825,0.6790,0.6503,0.2519,0.2829,0.6135,0.070
lr,Logistic Regression,0.6807,0.6713,0.6807,0.6760,0.6490,0.2487,0.2782,0.6122,1.790
ridge,Ridge Classifier,0.6811,0.6710,0.6811,0.6778,0.6475,0.2467,0.2788,0.6108,0.070


In [4]:
final_pycaret_model = finalize_model(best_model)
holdout_predictions = predict_model(final_pycaret_model, data=X_test.copy())
pycaret_labels = holdout_predictions["prediction_label"]
pycaret_results = pd.DataFrame([
    {
        "model": "PyCaret selected model",
        "accuracy": accuracy_score(y_test, pycaret_labels),
        "balanced_accuracy": balanced_accuracy_score(y_test, pycaret_labels),
        "f1_poisonous": f1_score(y_test, pycaret_labels, pos_label="p"),
    }
])
display(pd.concat([baseline_results, pycaret_results], ignore_index=True))


,model,accuracy,balanced_accuracy,f1_poisonous
0,Majority class,0.621,0.500000,0.000000
1,PyCaret selected model,0.796,0.758633,0.691843


## Conclusion

PyCaret selected Random Forest as the strongest candidate. We now tune it using cross-validation on the training data and validate the finalized model once on the holdout set. The model is the leading choice because it outperformed the majority baseline and achieved the best balanced accuracy in model comparison. Final adoption should depend especially on poisonous recall, since missing a poisonous mushroom is more serious than incorrectly rejecting an edible one. The validation metrics and finalized model are saved for reproducibility.


## Random Forest training, tuning, and validation

The Random Forest is tuned using cross-validation on the training data only. The holdout test set is used once for final validation, and the metrics are saved for later comparison.


In [5]:
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, roc_auc_score
from pycaret.classification import save_model, tune_model

tuned_rf = tune_model(
    best_model,
    optimize="Balanced Accuracy",
    fold=5,
    choose_better=True,
    verbose=False,
)
final_rf = finalize_model(tuned_rf)
rf_holdout = predict_model(final_rf, data=X_test.copy())
rf_labels = rf_holdout["prediction_label"]
rf_classes = list(final_rf.classes_)
rf_positive_index = rf_classes.index("p") if "p" in rf_classes else 1
rf_probabilities = final_rf.predict_proba(X_test)[:, rf_positive_index]

rf_report = classification_report(
    y_test, rf_labels, target_names=["edible", "poisonous"], output_dict=True
)
rf_metrics = {
    "model": "Tuned Random Forest",
    "accuracy": accuracy_score(y_test, rf_labels),
    "balanced_accuracy": balanced_accuracy_score(y_test, rf_labels),
    "poisonous_precision": precision_score(y_test, rf_labels, pos_label="p"),
    "poisonous_recall": recall_score(y_test, rf_labels, pos_label="p"),
    "poisonous_f1": f1_score(y_test, rf_labels, pos_label="p"),
    "roc_auc": roc_auc_score((y_test == "p").astype(int), rf_probabilities),
}

print(pd.Series(rf_metrics))
print("\nConfusion matrix [edible, poisonous]:\n", confusion_matrix(y_test, rf_labels))

metrics_path = Path("random_forest_metrics.csv")
pd.DataFrame([rf_metrics]).to_csv(metrics_path, index=False)
Path("random_forest_metrics.json").write_text(
    pd.Series({**rf_metrics, "classification_report": rf_report}).to_json(indent=2),
    encoding="utf-8",
)
save_model(final_rf, "random_forest_final")
print(f"Saved metrics to {metrics_path} and random_forest_metrics.json")
print("Saved model to random_forest_final.pkl")


model                  Tuned Random Forest
accuracy                             0.796
balanced_accuracy                 0.758633
poisonous_precision               0.809187
poisonous_recall                  0.604222
poisonous_f1                      0.691843
roc_auc                           0.838757
dtype: object

Confusion matrix [edible, poisonous]:
 [[567  54]
 [150 229]]
Transformation Pipeline and Model Successfully Saved
Saved metrics to random_forest_metrics.csv and random_forest_metrics.json
Saved model to random_forest_final.pkl


## XGBoost training, tuning, and validation

XGBoost is tuned with randomized cross-validation on the training data only. The holdout test set is used once for final validation, and metrics plus the fitted model are saved for comparison with Random Forest.


In [6]:
from pathlib import Path
from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

xgb_numeric = X_train.select_dtypes(include="number").columns.tolist()
xgb_categorical = X_train.select_dtypes(exclude="number").columns.tolist()
xgb_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median", add_indicator=True), xgb_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), xgb_categorical),
])

xgb_pipeline = Pipeline([
    ("preprocessor", xgb_preprocessor),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

xgb_parameters = {
    "model__n_estimators": [150, 250, 400],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.03, 0.08, 0.15],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
}
xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=xgb_parameters,
    n_iter=15,
    scoring="balanced_accuracy",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
)
xgb_search.fit(X_train, (y_train == "p").astype(int))
xgb_final = xgb_search.best_estimator_
xgb_labels = np.where(xgb_final.predict(X_test) == 1, "p", "e")
xgb_probabilities = xgb_final.predict_proba(X_test)[:, 1]
xgb_report = classification_report(
    y_test, xgb_labels, target_names=["edible", "poisonous"], output_dict=True
)
xgb_metrics = {
    "model": "Tuned XGBoost",
    "accuracy": accuracy_score(y_test, xgb_labels),
    "balanced_accuracy": balanced_accuracy_score(y_test, xgb_labels),
    "poisonous_precision": precision_score(y_test, xgb_labels, pos_label="p"),
    "poisonous_recall": recall_score(y_test, xgb_labels, pos_label="p"),
    "poisonous_f1": f1_score(y_test, xgb_labels, pos_label="p"),
    "roc_auc": roc_auc_score((y_test == "p").astype(int), xgb_probabilities),
}

print("Best parameters:", xgb_search.best_params_)
print(pd.Series(xgb_metrics))
print("\nConfusion matrix [edible, poisonous]:\n", confusion_matrix(y_test, xgb_labels))

pd.DataFrame([xgb_metrics]).to_csv("xgboost_metrics.csv", index=False)
Path("xgboost_metrics.json").write_text(
    pd.Series({
        **xgb_metrics,
        "best_cv_balanced_accuracy": xgb_search.best_score_,
        "best_parameters": xgb_search.best_params_,
        "classification_report": xgb_report,
    }).to_json(indent=2),
    encoding="utf-8",
)
dump(xgb_final, "xgboost_final.joblib")
print("Saved metrics to xgboost_metrics.csv and xgboost_metrics.json")
print("Saved model to xgboost_final.joblib")


Best parameters: {'model__subsample': 0.8, 'model__n_estimators': 250, 'model__max_depth': 5, 'model__learning_rate': 0.08, 'model__colsample_bytree': 1.0}
model                  Tuned XGBoost
accuracy                        0.81
balanced_accuracy           0.771447
poisonous_precision         0.843636
poisonous_recall            0.612137
poisonous_f1                 0.70948
roc_auc                     0.842203
dtype: object

Confusion matrix [edible, poisonous]:
 [[578  43]
 [147 232]]
Saved metrics to xgboost_metrics.csv and xgboost_metrics.json
Saved model to xgboost_final.joblib
